In [1]:
# ============================================================
# MODULE 4: DATA CLEANING & PREPROCESSING
# NYC Taxi Trip Duration & Congestion Pricing ML Project
# ============================================================

# Input: Raw data from yellow_tripdata_2025-01.parquet
# Output: Two cleaned datasets for Model 1 (Duration) & Model 2 (Congestion)
# ============================================================

In [2]:
# ============================================================
# STEP 0: IMPORT LIBRARIES & LOAD RAW DATA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [4]:
# Load raw data
file_path = r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\raw\yellow_tripdata_2025-01.parquet"
df = pd.read_parquet(file_path)

In [5]:
# Document starting point (before snapshot)
print("=" * 70)
print("STARTING POINT: RAW DATA")
print("=" * 70)
print(f"Raw data shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")

STARTING POINT: RAW DATA
Raw data shape: (3475226, 20)

Null counts:
VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          540149
trip_distance                 0
RatecodeID               540149
store_and_fwd_flag       540149
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     540149
Airport_fee              540149
cbd_congestion_fee            0
dtype: int64


In [6]:
# ============================================================
# STEP 1: DROP ROWS WITH MISSING VALUES
# ============================================================
# 540,149 rows have nulls across 5 columns:
# passenger_count, RatecodeID, store_and_fwd_flag,
# congestion_surcharge, Airport_fee
# Decision: DROP (not impute) because:
#   - 2.93M rows remain after dropping (plenty for ML)
#   - Imputing 540K values would introduce artificial bias
#   - All nulls are in the SAME rows (vendor data issue)
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: DROP ROWS WITH MISSING VALUES")
print("=" * 70)

before = df.shape[0]
df = df.dropna()

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")
print(f"Remaining nulls: {df.isnull().sum().sum()}")



STEP 1: DROP ROWS WITH MISSING VALUES
Rows removed: 540,149
Remaining rows: 2,935,077
Remaining nulls: 0


In [7]:
# ============================================================
# STEPS 2 & 3: REMOVE INVALID PASSENGER COUNTS
# ============================================================
# Step 2: Remove passenger_count == 0 (impossible - no passengers)
# Step 3: Remove passenger_count 7-9 (exceeds taxi capacity)
# Combined into one filter: keep only 1-6 passengers
# ============================================================

print("\n" + "=" * 70)
print("STEPS 2 & 3: REMOVE INVALID PASSENGER COUNTS (0 and 7-9)")
print("=" * 70)

before = df.shape[0]
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEPS 2 & 3: REMOVE INVALID PASSENGER COUNTS (0 and 7-9)
Rows removed: 24,674
Remaining rows: 2,910,403


In [8]:
# ============================================================
# STEP 4: REMOVE ZERO TRIP DISTANCE
# ============================================================
# Trips with 0 distance are invalid — the taxi didn't move
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: REMOVE ZERO TRIP DISTANCE")
print("=" * 70)

before = df.shape[0]
df = df[df['trip_distance'] > 0]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEP 4: REMOVE ZERO TRIP DISTANCE
Rows removed: 37,902
Remaining rows: 2,872,501


In [9]:
# ============================================================
# STEP 5: REMOVE EXTREME TRIP DISTANCE (> 100 MILES)
# ============================================================
# NYC taxi trips realistically don't exceed 100 miles
# Extreme values like 276,000+ miles are data errors
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: REMOVE EXTREME TRIP DISTANCE (> 100 MILES)")
print("=" * 70)

before = df.shape[0]
df = df[df['trip_distance'] < 100]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEP 5: REMOVE EXTREME TRIP DISTANCE (> 100 MILES)
Rows removed: 49
Remaining rows: 2,872,452


In [10]:
# ============================================================
# STEPS 6 & 7: REMOVE NEGATIVE AND EXTREME FARES
# ============================================================
# Step 6: Remove negative fare_amount (refunds/errors)
# Step 7: Remove extreme fare_amount (> $1,000)
# Combined into one filter: keep fares between $0 and $1,000
# ============================================================

print("\n" + "=" * 70)
print("STEPS 6 & 7: REMOVE NEGATIVE AND EXTREME FARES")
print("=" * 70)

before = df.shape[0]
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] <= 1000)]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEPS 6 & 7: REMOVE NEGATIVE AND EXTREME FARES
Rows removed: 55,674
Remaining rows: 2,816,778


In [11]:
# ============================================================
# STEP 8: REMOVE IMPOSSIBLE TIMESTAMPS
# ============================================================
# Remove trips where dropoff is before or equal to pickup
# Using strict < also removes zero-duration trips (pickup == dropoff)
# which are physically impossible taxi trips
# ============================================================

print("\n" + "=" * 70)
print("STEP 8: REMOVE IMPOSSIBLE TIMESTAMPS")
print("=" * 70)

before = df.shape[0]
df = df[df['tpep_pickup_datetime'] < df['tpep_dropoff_datetime']]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEP 8: REMOVE IMPOSSIBLE TIMESTAMPS
Rows removed: 1,221
Remaining rows: 2,815,557


In [12]:
# ============================================================
# STEP 11: REMOVE RATECODE ID = 99 (NULL/UNKNOWN)
# ============================================================
# RatecodeID = 99 means "null/unknown" per TLC data dictionary
# These trips have unreliable rate information
# Applied BEFORE split — affects both models
# ============================================================

print("\n" + "=" * 70)
print("STEP 11: REMOVE RATECODE ID = 99 (NULL/UNKNOWN)")
print("=" * 70)

before = df.shape[0]
df = df[df['RatecodeID'] != 99]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEP 11: REMOVE RATECODE ID = 99 (NULL/UNKNOWN)
Rows removed: 39,004
Remaining rows: 2,776,553


In [13]:
# ============================================================
# STEP 12: REMOVE INVALID PAYMENT TYPES
# ============================================================
# Keep only valid payment types:
#   0 = Flex Fare trip (verified from TLC Data Dictionary March 2025)
#   1 = Credit card
#   2 = Cash
# Remove:
#   3 = No charge (not a real transaction)
#   4 = Dispute (contested trip)
#   5 = Unknown
# Applied BEFORE split — affects both models
# ============================================================

print("\n" + "=" * 70)
print("STEP 12: REMOVE INVALID PAYMENT TYPES (3, 4, 5)")
print("=" * 70)

before = df.shape[0]
df = df[df['payment_type'].isin([0, 1, 2])]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


STEP 12: REMOVE INVALID PAYMENT TYPES (3, 4, 5)
Rows removed: 47,223
Remaining rows: 2,729,330


In [14]:
# ============================================================
# STEP 13: CALCULATE TRIP DURATION (REGRESSION TARGET)
# ============================================================
# Formula: (dropoff_time - pickup_time) converted to minutes
# This is the TARGET VARIABLE for Model 1 (Duration Regression)
# ============================================================

print("\n" + "=" * 70)
print("STEP 13: CALCULATE TRIP DURATION")
print("=" * 70)

df['trip_duration_min'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60

print(f"Duration range: {df['trip_duration_min'].min():.2f} to {df['trip_duration_min'].max():.2f} min")
print(f"Mean duration: {df['trip_duration_min'].mean():.2f} min")


STEP 13: CALCULATE TRIP DURATION
Duration range: 0.02 to 5626.32 min
Mean duration: 14.68 min


In [15]:
# ============================================================
# STEP 14: FILTER DURATION OUTLIERS (1-180 MINUTES)
# ============================================================
# Remove trips < 1 min (too short to be real)
# Remove trips > 180 min / 3 hours (too long for typical NYC taxi)
# ============================================================

print("\n" + "=" * 70)
print("STEP 14: FILTER DURATION OUTLIERS (1-180 MINUTES)")
print("=" * 70)

before = df.shape[0]
df = df[(df['trip_duration_min'] >= 1) & (df['trip_duration_min'] <= 180)]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")
print(f"New duration range: {df['trip_duration_min'].min():.2f} to {df['trip_duration_min'].max():.2f} min")


STEP 14: FILTER DURATION OUTLIERS (1-180 MINUTES)
Rows removed: 7,823
Remaining rows: 2,721,507
New duration range: 1.00 to 179.03 min


In [16]:
# ============================================================
# STEP 15: CLEAN TOTAL AMOUNT (> $600)
# ============================================================
# Remove extreme total_amount values
# Total amount includes fare + extras + tips + tolls + surcharges
# $600 is a generous upper limit for any NYC taxi trip
# ============================================================

print("\n" + "=" * 70)
print("STEP 15: CLEAN TOTAL AMOUNT (> $600)")
print("=" * 70)

before = df.shape[0]
df = df[(df['total_amount'] > 0) & (df['total_amount'] <= 600)]

print(f"Rows removed: {before - df.shape[0]:,}")
print(f"Remaining rows: {df.shape[0]:,}")


# ============================================================
# STEP 16: CHECK FOR DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("STEP 16: DUPLICATE CHECK")
print("=" * 70)

print(f"Duplicate rows: {df.duplicated().sum()}")


STEP 15: CLEAN TOTAL AMOUNT (> $600)
Rows removed: 6
Remaining rows: 2,721,501

STEP 16: DUPLICATE CHECK
Duplicate rows: 0


In [17]:
# ============================================================
# STEP 17: CREATE CONGESTION FEE TARGET (CLASSIFICATION)
# ============================================================
# Convert cbd_congestion_fee to binary:
#   $0.75  → 1 (YES, fee charged)
#   $0.00  → 0 (NO, no fee)
#   -$0.75 → 0 (refund, treated as no fee)
# This is the TARGET VARIABLE for Model 2 (Classification)
# ============================================================

print("\n" + "=" * 70)
print("STEP 17: CREATE CONGESTION FEE BINARY TARGET")
print("=" * 70)

df['has_congestion_fee'] = (df['cbd_congestion_fee'] > 0).astype(int)

print("Distribution:")
print(df['has_congestion_fee'].value_counts())
fee_pct = df['has_congestion_fee'].mean() * 100
print(f"\nFee rate: {fee_pct:.1f}% of trips have congestion fee")


STEP 17: CREATE CONGESTION FEE BINARY TARGET
Distribution:
has_congestion_fee
1    1798839
0     922662
Name: count, dtype: int64

Fee rate: 66.1% of trips have congestion fee


In [18]:
# ============================================================
# STEP 18: SPLIT INTO MODEL-SPECIFIC DATAFRAMES
# ============================================================
# Model 1 (Duration Regression): Keeps ALL dates — more training data
# Model 2 (Congestion Classification): Removes pre-Jan-5 and refunds
#   because congestion pricing started Jan 5, 2025
# ============================================================

print("\n" + "=" * 70)
print("STEP 18: SPLIT INTO MODEL-SPECIFIC DATAFRAMES")
print("=" * 70)

df_model1 = df.copy()
df_model2 = df.copy()

# Step 9: Remove congestion fee refunds (Model 2 only)
before = df_model2.shape[0]
df_model2 = df_model2[df_model2['cbd_congestion_fee'] != -0.75]
print(f"Step 9 (Model 2) - Refund rows removed: {before - df_model2.shape[0]:,}")

# Step 10: Remove pre-Jan-5 trips (Model 2 only)
before = df_model2.shape[0]
df_model2 = df_model2[df_model2['tpep_pickup_datetime'] >= '2025-01-05']
print(f"Step 10 (Model 2) - Pre-Jan-5 rows removed: {before - df_model2.shape[0]:,}")

print(f"\nModel 1 (Duration): {df_model1.shape}")
print(f"Model 2 (Congestion): {df_model2.shape}")


STEP 18: SPLIT INTO MODEL-SPECIFIC DATAFRAMES
Step 9 (Model 2) - Refund rows removed: 0
Step 10 (Model 2) - Pre-Jan-5 rows removed: 309,075

Model 1 (Duration): (2721501, 22)
Model 2 (Congestion): (2412426, 22)


In [19]:
# ============================================================
# STEP 19: FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

print("\n=== Model 1 Verification ===")
print(f"Nulls: {df_model1.isnull().sum().sum()}")
print(f"Duplicates: {df_model1.duplicated().sum()}")
print(f"Zero passengers: {(df_model1['passenger_count'] == 0).sum()}")
print(f"Zero distance: {(df_model1['trip_distance'] == 0).sum()}")
print(f"Negative fares: {(df_model1['fare_amount'] <= 0).sum()}")
print(f"RatecodeID 99: {(df_model1['RatecodeID'] == 99).sum()}")
print(f"Duration range: {df_model1['trip_duration_min'].min():.2f} to {df_model1['trip_duration_min'].max():.2f} min")

print("\n=== Model 2 Verification ===")
print(f"Nulls: {df_model2.isnull().sum().sum()}")
print(f"Duplicates: {df_model2.duplicated().sum()}")
print(f"Pre-Jan-5 trips: {(df_model2['tpep_pickup_datetime'] < '2025-01-05').sum()}")
print(f"Congestion refunds: {(df_model2['cbd_congestion_fee'] == -0.75).sum()}")
print(f"has_congestion_fee distribution:\n{df_model2['has_congestion_fee'].value_counts()}")


FINAL VERIFICATION

=== Model 1 Verification ===
Nulls: 0
Duplicates: 0
Zero passengers: 0
Zero distance: 0
Negative fares: 0
RatecodeID 99: 0
Duration range: 1.00 to 179.03 min

=== Model 2 Verification ===
Nulls: 0
Duplicates: 0
Pre-Jan-5 trips: 0
Congestion refunds: 0
has_congestion_fee distribution:
has_congestion_fee
1    1798485
0     613941
Name: count, dtype: int64


In [20]:
# ============================================================
# STEP 20: SAVE CLEANED DATASETS
# ============================================================

print("\n" + "=" * 70)
print("SAVING CLEANED DATASETS")
print("=" * 70)

df_model1.to_parquet(
    r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\cleaned_model1_duration.parquet",
    index=False
)
df_model2.to_parquet(
    r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\cleaned_model2_congestion.parquet",
    index=False
)

print(f"\n✅ Both datasets saved!")
print(f"Model 1 (Duration):   {df_model1.shape[0]:,} rows, {df_model1.shape[1]} columns")
print(f"Model 2 (Congestion): {df_model2.shape[0]:,} rows, {df_model2.shape[1]} columns")
print(f"\nTotal removed from raw: {3475226 - df_model1.shape[0]:,} (Model 1), {3475226 - df_model2.shape[0]:,} (Model 2)")

print("\n" + "=" * 70)
print("DATA CLEANING COMPLETE!")
print("=" * 70)


SAVING CLEANED DATASETS

✅ Both datasets saved!
Model 1 (Duration):   2,721,501 rows, 22 columns
Model 2 (Congestion): 2,412,426 rows, 22 columns

Total removed from raw: 753,725 (Model 1), 1,062,800 (Model 2)

DATA CLEANING COMPLETE!
